
# Design a UUID Generator

A UUID Generator generates globally unique identifiers.

UUID stands for:

```text
Universally Unique Identifier
```

Example UUID:

```text
550e8400-e29b-41d4-a716-446655440000
```

Used in:
- databases
- distributed systems
- microservices
- transaction IDs
- session IDs

---

# 1. Problem Statement

We need a system that generates:

```text
Globally unique IDs
```

Requirements:
- No collisions
- Fast generation
- Distributed
- Highly scalable
- Fault tolerant

---

# Why Not Auto Increment IDs?

Simple DB auto increment:

```sql
1
2
3
4
```

Problems:
- Central bottleneck
- Single point of failure
- Not distributed
- Predictable IDs

---

# 2. UUID Basics

UUID is:

```text
128-bit identifier
```

Format:

```text
8-4-4-4-12
```

Example:

```text
550e8400-e29b-41d4-a716-446655440000
```

---

# UUID Versions

| Version | Based On |
|---|---|
| v1 | timestamp + MAC |
| v2 | DCE security |
| v3 | MD5 hash |
| v4 | Random |
| v5 | SHA1 hash |

Most common:
- UUID v4
- Snowflake IDs

---

# 3. UUID v4 Deep Dive

UUID v4 uses:

```text
Randomness
```

Structure:
- 128 bits total
- 122 random bits
- Remaining reserved

---

# Collision Probability

Even billions of UUIDs generated per second have extremely low collision probability because:

```text
2^122 possibilities
```

---

# UUID v4 Generation Flow

```text
1. Generate random bits
2. Set version bits
3. Set variant bits
4. Format UUID
5. Return ID
```

---

# Python Example

```python
import uuid

id = uuid.uuid4()

print(id)
```

---

# 4. Distributed UUID Generation

Goal:

```text
Each server independently generates IDs
```

without coordination.

---

# Architecture

```text
Clients
   ↓
Service A
Service B
Service C

Each independently generates UUIDs
```

---

# 5. Problem with UUID v4

UUIDs are random.

This causes:
- random disk writes
- page splits
- index fragmentation

Especially in:
- MySQL
- PostgreSQL

---

# 6. Ordered ID Requirement

Many companies need:

```text
Time sortable IDs
```

because:
- DB efficiency
- chronological ordering
- easier debugging

---

# 7. Snowflake IDs

Popularized by Twitter/X.

Snowflake creates:

```text
64-bit IDs
```

---

# Snowflake Structure

| Bits | Purpose |
|---|---|
| 41 | Timestamp |
| 10 | Machine ID |
| 12 | Sequence Number |

---

# Timestamp (41 bits)

Stores current timestamp.

Usually:
- milliseconds since custom epoch

---

# Machine ID (10 bits)

Supports:

```text
1024 machines
```

because:

```text
2^10 = 1024
```

---

# Sequence Number (12 bits)

Supports:

```text
4096 IDs/ms
```

because:

```text
2^12 = 4096
```

---

# Total Throughput

Per server:

```text
≈ 4 million IDs/sec
```

---

# Snowflake Generation Flow

```text
1. Get current timestamp
2. Read machine ID
3. Increment sequence number
4. Combine bits
5. Return ID
```

---

# Visualization

```text
| Timestamp | Machine ID | Sequence |
```

---

# 8. Distributed System Problems

## Problem 1 — Clock Goes Backward

Issue:
- duplicate IDs possible

### Solutions
- Wait until clock catches up
- Logical clocks
- NTP synchronization

---

## Problem 2 — Sequence Overflow

Suppose:

```text
5000 requests in same ms
```

But sequence supports only:

```text
4096
```

### Solution
- Wait for next millisecond

---

## Problem 3 — Machine ID Duplication

Two servers accidentally use same machine ID.

### Solutions
- ZooKeeper
- etcd
- Kubernetes StatefulSets
- Config service

---

# 9. Multi-Region Design

Need globally unique IDs across:
- US region
- India region
- Europe region

Solution:
- add Region ID bits

---

# Extended Snowflake Structure

| Bits | Purpose |
|---|---|
| Timestamp | Time |
| Region ID | Region |
| Machine ID | Server |
| Sequence | Counter |

---

# 10. Fault Tolerance

If one UUID node crashes:
- others continue generating

No SPOF.

---

# 11. Scalability

```text
Load Balancer
     ↓
UUID Node 1
UUID Node 2
UUID Node 3
```

Horizontal scaling.

---

# 12. Security Concerns

Snowflake IDs expose:
- timestamps
- traffic volume

UUID v4 safer because:
- random
- unpredictable

---

# 13. Tradeoffs

| Approach | Pros | Cons |
|---|---|---|
| UUID v4 | Simple, distributed | Random ordering |
| Snowflake | Ordered, compact | Clock dependency |
| Auto Increment | Simple | Central bottleneck |

---

# 14. Final Production Architecture

```text
Clients
   ↓
Load Balancer
   ↓
UUID Generator Cluster
   ↓
Generated IDs
```

Each node:
- independently generates IDs
- no DB coordination
- horizontally scalable

---

# 15. Final Interview Summary

For distributed systems requiring globally unique identifiers:
- use UUID v4 for simplicity and decentralized generation
- use Snowflake IDs when ordered IDs and DB efficiency are important

Handle:
- clock synchronization
- machine ID allocation
- sequence overflow
for reliable distributed ID generation at scale.
